<a id="sec-presentacion"></a>

# Detección de intrusiones MQTT — MQTT_UAD (cuatro clases)

Este notebook entrena **tres modelos supervisados multiclase** para MQTT sin
cifrar: XGBoost, LSTM supervisado e híbrido LSTM + XGBoost. La clase de un frame
o de la última fila de una ventana es una de
`['normal', 'DoS', 'mitm', 'intrusion']`. Además se conserva un **autoencoder
LSTM auxiliar** entrenado solo con ventanas completamente normales: su MSE
alimenta al híbrido como feature continua y sirve de referencia binaria, pero no
es un modo público multiclase.

Se conservan las rutas originales de entrada y salida.
Las features combinan red/transporte y campos MQTT numéricos, entropía de payload
y estructura de topic. IP, MAC, client_id, credenciales, puertos efímeros y tiempos
absolutos/relativos de captura solo sirven de contexto o se excluyen.

La captura y el agente admiten tramas Ethernet con `frame.time_epoch`. Los frames
IPv4/TCP se agrupan mediante endpoints y CONNECT; los demás frames L2 se agrupan
por par MAC no dirigido y episodios causales separados por más de 1 segundo. Las
ventanas siempre son direccionales. Las particiones separan conexiones/episodios
completos y las ventanas no cruzan archivos, grupos, direcciones ni particiones.
Los CSV no tienen `tcp.stream`, por lo que ambos tipos de grupo son aproximaciones.
No es una prueba de generalización a capturas independientes ni a MQTTS.

## Índice

> Este notebook se ejecuta **de arriba abajo en un kernel limpio**. El índice es
> una guía de lectura; no conviene ejecutar secciones aisladas. La ejecución
> completa y la validación las realiza el usuario.

- [Presentación](#sec-presentacion)
- [Preparación: importaciones y parámetros](#sec-preparacion)
- [1. Carga y contexto de las capturas](#sec-carga)
- [2. Features MQTT y diagnóstico de columnas](#sec-features)
- [3. Particiones agrupadas y temporales](#sec-particiones)
- [4. Estado de features ajustado por partición](#sec-estado)
- [5. Modelos, entrenamiento y helpers](#sec-modelos)
- [6. Métricas multiclase y cobertura](#sec-metricas)
- [7. Evaluación de cinco folds con MSE out-of-fold](#sec-evaluacion)
- [8. Evaluación temporal adicional](#sec-temporal)
- [9. Paquete final y exportación](#sec-exportacion)


<a id="sec-preparacion"></a>
## Preparación: importaciones y parámetros

Se cargan las dependencias (numpy, pandas, torch, xgboost, sklearn) y se fijan
semilla, longitud de secuencia, número de folds, relleno de NaN, ventana de tasa
de bytes y el dispositivo de cómputo. **Esta celda solo importa y define
constantes; no entrena ni escribe artefactos.**


In [1]:
import os
import json
import math
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             balanced_accuracy_score, f1_score, roc_auc_score)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = sns = None  # Los reportes numéricos siguen funcionando sin librerías de plots.

SEED = 42
SEQ_LEN = 10
N_SPLITS = 5
NAN_FILL = -1.0
BATCH_SIZE = 512
EPOCHS = 10
DIRECTION_RATE_WINDOW_SECONDS = 5.0
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
XGB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch: {DEVICE}; XGBoost: {XGB_DEVICE}')


Torch: cuda; XGBoost: cuda


<a id="sec-carga"></a>

## 1. Carga y contexto de las capturas

No se asignan etiquetas de ataque a un archivo completo: se conserva `type` por frame.


### Rutas, etiquetas y contratos de carga

Se declaran las rutas originales de Kaggle, la columna objetivo `type`, el mapa
`LABEL_MAP`, el gap de episodios L2 y las columnas de flujo TCP y de capa 2.
Solo son definiciones.


In [2]:
FILES = {
    'DoS':       '/kaggle/input/datasets/alejandropenagos84/dataset-mqtt-a/DoS.csv',
    'MitM':      '/kaggle/input/datasets/alejandropenagos84/dataset-mqtt-a/MitM.csv',
    'Intrusion': '/kaggle/input/datasets/alejandropenagos84/dataset-mqtt-a/Intrusion.csv',
}
TARGET_COL = 'type'
L2_EPISODE_GAP_SECONDS = 1.0

LABEL_MAP = {'dos': 'DoS', 'mitm': 'mitm', 'intrusion': 'intrusion', 'normal': 'normal'}
FLOW_COLUMNS = ['ip.src', 'tcp.srcport', 'ip.dst', 'tcp.dstport']
L2_COLUMNS = ['eth.src', 'eth.dst']



### Definición de `load_and_merge`

`load_and_merge` valida columnas, normaliza etiquetas, descarta filas sin reloj o
sin contexto Ethernet/TCP, asigna `_connection`, `_direction` y `_observation`, y
concatena las capturas. **Esta celda define la función; no lee archivos.**


In [3]:
def load_and_merge(files):
    frames = []
    for capture, path in files.items():
        data = pd.read_csv(path, low_memory=False, dtype={
            'mqtt.msg': 'string', 'mqtt.topic': 'string',
            'ip.src': 'string', 'ip.dst': 'string',
            'eth.src': 'string', 'eth.dst': 'string',
        })
        if 'Type' in data.columns and TARGET_COL not in data.columns:
            data = data.rename(columns={'Type': TARGET_COL})
        required = FLOW_COLUMNS + L2_COLUMNS + [
            'frame.time_epoch', 'frame.time_delta', 'frame.time_delta_displayed',
            'frame.len', 'frame.cap_len', 'mqtt.msgtype', TARGET_COL,
        ]
        missing = set(required) - set(data.columns)
        if missing:
            raise ValueError(f'{capture}: faltan columnas {sorted(missing)}')
        labels = data[TARGET_COL].astype(str).str.strip().str.lower()
        if not labels.isin(LABEL_MAP).all():
            raise ValueError(f'{capture}: etiquetas desconocidas {labels[~labels.isin(LABEL_MAP)].unique()}')
        data[TARGET_COL] = labels.map(LABEL_MAP)
        for col in ('tcp.srcport', 'tcp.dstport', 'frame.time_epoch', 'mqtt.msgtype'):
            data[col] = pd.to_numeric(data[col], errors='coerce')
        for col in L2_COLUMNS:
            data[col] = data[col].astype('string').str.strip().str.lower()
        print(f'{capture}: todas las tramas: {data[TARGET_COL].value_counts().to_dict()}')

        valid_epoch = data['frame.time_epoch'].notna() & np.isfinite(data['frame.time_epoch'])
        valid_tcp = (
            valid_epoch
            & data['ip.src'].fillna('').str.strip().ne('')
            & data['ip.dst'].fillna('').str.strip().ne('')
            & data['tcp.srcport'].between(1, 65535)
            & data['tcp.dstport'].between(1, 65535)
            & data['tcp.srcport'].mod(1).eq(0)
            & data['tcp.dstport'].mod(1).eq(0)
        )
        valid_l2 = (
            valid_epoch
            & data['eth.src'].fillna('').str.strip().ne('')
            & data['eth.dst'].fillna('').str.strip().ne('')
        )
        observable = valid_tcp | valid_l2
        discarded = ~observable
        if discarded.any():
            causas = {
                'sin_epoch': ~valid_epoch,
                'sin_ip_src': data['ip.src'].fillna('').str.strip().eq(''),
                'sin_ip_dst': data['ip.dst'].fillna('').str.strip().eq(''),
                'sin_puerto_src': ~data['tcp.srcport'].between(1, 65535),
                'sin_puerto_dst': ~data['tcp.dstport'].between(1, 65535),
                'sin_mac': ~(data['eth.src'].fillna('').str.strip().ne('')
                             & data['eth.dst'].fillna('').str.strip().ne('')),
            }
            resumen = {name: int((discarded & mask).sum())
                       for name, mask in causas.items()}
            print(f'{capture}: descartadas por causa (solapan): {resumen}')
        before = len(data)
        data = data.loc[observable].copy()
        data['_original_row'] = data.index
        data = data.sort_values('frame.time_epoch', kind='stable')
        data['_capture'] = capture
        data['_observation'] = np.where(valid_tcp.loc[data.index], 'tcp', 'l2')

        connections = [None] * len(data)
        directions = [None] * len(data)
        endpoints = [None] * len(data)

        tcp_positions = np.flatnonzero(data['_observation'].eq('tcp').to_numpy())
        if len(tcp_positions):
            tcp_data = data.iloc[tcp_positions]
            tcp_endpoints = [
                tuple(sorted(((str(src), int(sport)), (str(dst), int(dport)))))
                for src, sport, dst, dport
                in tcp_data[FLOW_COLUMNS].itertuples(index=False, name=None)
            ]
            endpoint_series = pd.Series(tcp_endpoints, index=tcp_data.index, dtype=object)
            connect = tcp_data['mqtt.msgtype'].eq(1).astype(int)
            sessions = connect.groupby(endpoint_series, sort=False).cumsum()
            for pos, endpoint, session, flow in zip(
                    tcp_positions, tcp_endpoints, sessions,
                    tcp_data[FLOW_COLUMNS].itertuples(index=False, name=None)):
                connection = (capture, 'tcp', endpoint, int(session))
                src, sport, dst, dport = flow
                endpoints[pos] = endpoint
                connections[pos] = connection
                directions[pos] = (connection, str(src), int(sport), str(dst), int(dport))

        l2_positions = np.flatnonzero(data['_observation'].eq('l2').to_numpy())
        if len(l2_positions):
            l2_data = data.iloc[l2_positions]
            l2_pairs = [
                tuple(sorted((str(src), str(dst))))
                for src, dst in l2_data[L2_COLUMNS].itertuples(index=False, name=None)
            ]
            pair_series = pd.Series(l2_pairs, index=l2_data.index, dtype=object)
            gaps = l2_data['frame.time_epoch'].groupby(pair_series, sort=False).diff()
            episode_start = gaps.isna() | gaps.gt(L2_EPISODE_GAP_SECONDS)
            episodes = episode_start.groupby(pair_series, sort=False).cumsum()
            for pos, pair, episode, macs in zip(
                    l2_positions, l2_pairs, episodes,
                    l2_data[L2_COLUMNS].itertuples(index=False, name=None)):
                connection = (capture, 'l2', pair, int(episode))
                src, dst = macs
                endpoints[pos] = pair
                connections[pos] = connection
                directions[pos] = (connection, str(src), str(dst))

        data['_endpoints'] = pd.Series(endpoints, index=data.index, dtype=object)
        data['_connection'] = connections
        data['_direction'] = directions
        print(f'{capture}: excluidas sin contexto Ethernet/reloj: {before - len(data)}')
        print(f'{capture}: observación: {data["_observation"].value_counts().to_dict()}')
        print(f'{capture}: seleccionadas: {data[TARGET_COL].value_counts().to_dict()}')
        frames.append(data)
    merged = pd.concat(frames, ignore_index=True)
    if merged.empty:
        raise ValueError('No hay frames Ethernet observables con frame.time_epoch')
    return merged



### Lectura y resumen de capturas

Al ejecutar esta celda se leen los CSV, se imprime el diagnóstico por captura y
se construye `df_raw`, la única tabla base del resto del notebook. **Ejecutar.**


In [4]:
df_raw = load_and_merge(FILES)
print(f'Dataset Ethernet seleccionado: {df_raw.shape}')
print(df_raw[TARGET_COL].value_counts())

DoS: todas las tramas: {'normal': 49111, 'DoS': 45514}
DoS: excluidas sin contexto Ethernet/reloj: 0
DoS: observación: {'tcp': 88460, 'l2': 6165}
DoS: seleccionadas: {'normal': 49111, 'DoS': 45514}
MitM: todas las tramas: {'normal': 106813, 'mitm': 3855}
MitM: excluidas sin contexto Ethernet/reloj: 0
MitM: observación: {'tcp': 99514, 'l2': 11154}
MitM: seleccionadas: {'normal': 106813, 'mitm': 3855}
Intrusion: todas las tramas: {'normal': 78995, 'intrusion': 1898}
Intrusion: excluidas sin contexto Ethernet/reloj: 0
Intrusion: observación: {'tcp': 71757, 'l2': 9136}
Intrusion: seleccionadas: {'normal': 78995, 'intrusion': 1898}
Dataset Ethernet seleccionado: (286186, 73)
type
normal       234919
DoS           45514
mitm           3855
intrusion      1898
Name: count, dtype: int64


<a id="sec-features"></a>

## 2. Features MQTT y diagnostico de columnas

Se conservan **34 features**: red/transporte, campos MQTT con senal y derivadas de
entropia y ratio. Antes de construir la matriz se imprime el diagnostico por columna
cruda. Las tablas siguientes resumen, por grupo, que se conserva y que se descarta.

### Criterios de descarte

| Motivo | Criterio sobre los datos | Tratamiento |
|---|---|---|
| Columna vacia | 100% de valores nulos | Se descarta |
| Metadato de conexion | Mas del 99% nula y constante (solo aparece en CONNECT/CONNACK) | Se descarta |
| Identidad | IP, MAC o `client_id` | Se descarta |
| Fuga de captura | Reloj absoluto/relativo u orden de trama | Se descarta |
| Puerto efimero | `tcp.srcport`/`tcp.dstport` | Se resume en banderas `fe_is_*` |
| Duplicado exacto | Columna igual a otra en el 100% de las tramas | Se descarta |
| Derivables | Contenido ya codificado en `mqtt.hdrflags` | Se descarta |
| Texto crudo | `mqtt.msg`/`mqtt.topic` no numericos | Se resume en entropia/profundidad |
| Constante | Un unico valor en todas las tramas | Se descarta |

### Features conservadas (34)

| Grupo | Columnas | Que aportan |
|---|---|---|
| Red | `frame.len`, `frame.time_delta` | Tamano de trama y ritmo de envio |
| Red derivadas | `fe_bytes_per_sec`, `fe_is_standard_mqtt_port`, `fe_is_broker_to_client`, `fe_log_len` | Tasa causal de bytes por dirección (ventana 5 s, `log1p`), puerto MQTT y escala logaritmica |
| MQTT crudas | `mqtt.len`, `mqtt.topic_len` | Longitud MQTT y del topic |
| MQTT decodificadas (Etapa 1) | `fe_mqtt_type`, `fe_mqtt_qos`, `fe_mqtt_dup`, `fe_mqtt_retain`, `fe_topic_wildcard`, `fe_has_mqtt` | Tipo/QoS/DUP/RETAIN de `hdrflags`, wildcard y presencia de MQTT |
| MQTT derivadas | `fe_msg_entropy`, `fe_topic_entropy`, `fe_topic_depth`, `fe_payload_ratio` | Entropia y estructura del topic, ratio de payload |
| Contexto CONNECT (Etapa 1) | `fe_mqtt_ver`, `fe_mqtt_kalive`, `fe_mqtt_cleansess`, `fe_mqtt_clientid_len` | Version, keep-alive, clean session y longitud de clientid propagados del CONNECT |
| Identidad derivada (Etapa 2) | `fe_origen_conocido`, `fe_origen_observable` | Lookup train-only y si el frame trae IP/client ID; distingue desconocido de ausente |
| Contexto por origen (Etapa 2) | `fe_origen_frames_s`, `fe_origen_bytes_s`, `fe_origen_conexiones`, `fe_origen_connects` | Ventana causal de 5 s por `ip.src` |
| Contexto L2 (Etapa 2) | `fe_is_broadcast`, `fe_is_multicast`, `fe_l2_frames_s`, `fe_l2_broadcasts`, `fe_l2_burst_ratio` | Broadcast/multicast y ventana causal de 5 s por `eth.src` |
| Novedad de tópico (Etapa 2) | `fe_topic_novedad` | Tópico no observado en train (allowlist `known_topics`) |

### Columnas descartadas

| Columna(s) | Semantica | Motivo |
|---|---|---|
| `frame.time_invalid`, `frame.coloring_rule.name`, `frame.coloring_rule.string`, `frame.comment`, `frame.comment.expert`, `frame.file_off`, `frame.incomplete`, `frame.interface_id`, `frame.interface_name`, `frame.link_nr`, `frame.md5_hash` | Metadatos internos de tshark | 100% nulas |
| `mqtt.username`, `mqtt.username_len`, `mqtt.passwd`, `mqtt.passwd_len`, `mqtt.willmsg`, `mqtt.willmsg_len`, `mqtt.willtopic`, `mqtt.willtopic_len` | Credenciales y testamento MQTT | 100% nulas |
| `mqtt.sub.qos`, `mqtt.suback.qos` | QoS de SUBSCRIBE y SUBACK | Practicamente 100% nulas |
| `mqtt.conack.flags`, `mqtt.conack.flags.reserved`, `mqtt.conack.flags.sp`, `mqtt.conack.val`, `mqtt.conflag.cleansess`, `mqtt.conflag.passwd`, `mqtt.conflag.qos`, `mqtt.conflag.reserved`, `mqtt.conflag.retain`, `mqtt.conflag.uname`, `mqtt.conflag.willflag`, `mqtt.conflags`, `mqtt.kalive`, `mqtt.proto_len`, `mqtt.protoname`, `mqtt.ver` | Parametros de CONNECT y CONNACK | Mas del 99% nulas |
| `mqtt.msgid` | Identificador de paquete PUBLISH con QoS mayor que 0 | Mas del 99% nula |
| `mqtt.clientid`, `mqtt.clientid_len` | Identidad de cliente en CONNECT | Identidad |
| `ip.src`, `ip.dst`, `eth.src`, `eth.dst` | Identidad de host y de MAC | No describe comportamiento |
| `frame.time_epoch`, `frame.time_relative`, `frame.number` | Reloj y orden de captura | Fuga entre capturas |
| `tcp.srcport`, `tcp.dstport` | Puertos TCP efectivos | Leakage de puertos efimeros; solo banderas |
| `frame.cap_len` | Bytes capturados de la trama | Igual a `frame.len` en el 100% |
| `frame.time_delta_displayed` | Delta mostrado de tiempo | Igual a `frame.time_delta` en el 100% |
| `mqtt.msgtype`, `mqtt.qos`, `mqtt.retain`, `mqtt.dupflag` | Tipo y flags de PUBLISH | Derivables de `mqtt.hdrflags` |
| `mqtt.msg`, `mqtt.topic` | Payload y topic publicados | Texto crudo; se resumen en derivadas |
| `frame.encap_type`, `frame.ignored`, `frame.marked`, `frame.offset_shift` | Banderas internas de captura | Constantes, sin varianza |

La validacion de particiones (seccion 3) calcula las etiquetas de ventana por fold
una sola vez, de modo que no recorre el dataset en cada combinacion.

### Orden de construcción

- **Etapa 1, antes del split:** las 22 features causales del frame (18 de red/MQTT y 4 de contexto de CONNECT).
- **Etapa 2, después del split:** `fe_origen_conocido` (allowlist aprendida
  solo de frames normales de train) y `fe_origen_observable` (1 si el frame
  trae `ip.src` o `mqtt.clientid`, 0 si faltan ambos).

`ip.src` y `mqtt.clientid` solo construyen la clave interna del lookup; nunca se
entregan como columnas al modelo. `(0, 0)` es identidad ausente y `(0, 1)` un
origen observado pero desconocido; ninguna regla asigna una clase por ello.

### Tasa de bytes y representación del payload (Fase 4)

- `fe_bytes_per_sec` es `log1p(tasa)` de una tasa causal: suma de `frame.len` de
  frames anteriores de la misma conexión/episodio y dirección con tiempo en
  `[t-5, t]`, excluye el frame actual y divide entre 5. Sin historial vale 0. El
  estado vive fuera del buffer de 11 frames y se reinicia por
  conexión/episodio, captura/partición o retroceso de reloj.
- La entropía se calcula sobre el texto original de `mqtt.msg`; no se aplica
  `bytes.fromhex` por apariencia. Para este dataset el capturador ya entrega la
  representación textual de MQTT_UAD. Limitación documentada: bytes no UTF-8.


### Helpers de texto y catálogos de features

Define la entropía de Shannon sobre texto y los catálogos que clasifican cada
columna cruda o derivada (`KEEP_RAW`, `DERIVED_SOURCES`, `DERIVED_FEATURES`,
`RAZONES`) y las columnas internas. Solo definiciones.


In [5]:
def entropia(texto):
    if pd.isna(texto) or not str(texto):
        return np.nan
    text = str(texto)
    probs = [count / len(text) for count in Counter(text).values()]
    return -sum(p * math.log2(p) for p in probs)


# Features crudas conservadas.
KEEP_RAW = ['frame.len', 'frame.time_delta', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.topic_len']
# Columnas que no se conservan crudas pero alimentan una derivada.
DERIVED_SOURCES = {
    'tcp.dstport': 'fe_is_standard_mqtt_port',
    'tcp.srcport': 'fe_is_broker_to_client',
    'mqtt.msg': 'fe_msg_entropy',
    'mqtt.topic': 'fe_topic_entropy y fe_topic_depth',
}
# Features derivadas generadas y su destino.
DERIVED_FEATURES = {
    'fe_bytes_per_sec': ('CONSERVA', 'Tasa causal de bytes por conexion/direccion en 5 s, publicada como log1p.'),
    'fe_is_standard_mqtt_port': ('CONSERVA', 'Destino en 1883 o 8883.'),
    'fe_is_broker_to_client': ('CONSERVA', 'Origen en 1883 o 8883.'),
    'fe_log_len': ('CONSERVA', 'log1p(frame.len); escala para el LSTM.'),
    'fe_msg_entropy': ('CONSERVA', 'Entropia del payload.'),
    'fe_topic_entropy': ('CONSERVA', 'Entropia del topic.'),
    'fe_topic_depth': ('CONSERVA', 'Niveles del topic.'),
    'fe_payload_ratio': ('CONSERVA', 'Ratio mqtt.len / frame.len.'),
    'fe_origen_conocido': ('CONSERVA', 'Lookup train-only de IP/client_id normal.'),
    'fe_origen_observable': ('CONSERVA', '1 si el frame trae ip.src o mqtt.clientid; 0 si faltan ambos.'),
    'fe_mqtt_ver': ('CONSERVA', 'Version MQTT propagada del CONNECT (Etapa 1).'),
    'fe_mqtt_kalive': ('CONSERVA', 'Keep-alive propagado del CONNECT (Etapa 1).'),
    'fe_mqtt_cleansess': ('CONSERVA', 'Clean session propagada del CONNECT (Etapa 1).'),
    'fe_mqtt_clientid_len': ('CONSERVA', 'Longitud del clientid propagada del CONNECT (Etapa 1).'),
    'fe_origen_frames_s': ('CONSERVA', 'Frames/s de la misma ip.src en 5 s (Etapa 2).'),
    'fe_origen_bytes_s': ('CONSERVA', 'Bytes/s de la misma ip.src en 5 s (Etapa 2).'),
    'fe_origen_conexiones': ('CONSERVA', 'Conexiones distintas de la misma ip.src en 5 s (Etapa 2).'),
    'fe_origen_connects': ('CONSERVA', 'CONNECTs de la misma ip.src en 5 s (Etapa 2).'),
    'fe_is_broadcast': ('CONSERVA', 'eth.dst igual a ff:ff:ff:ff:ff:ff (Etapa 2).'),
    'fe_is_multicast': ('CONSERVA', 'bit multicast en eth.dst (Etapa 2).'),
    'fe_l2_frames_s': ('CONSERVA', 'Frames/s del mismo eth.src en 5 s (Etapa 2).'),
    'fe_l2_broadcasts': ('CONSERVA', 'Broadcasts del mismo eth.src en 5 s (Etapa 2).'),
    'fe_l2_burst_ratio': ('CONSERVA', 'Fraccion de deltas <100 us del mismo eth.src (Etapa 2).'),
    'fe_mqtt_type': ('CONSERVA', 'Tipo de paquete MQTT decodificado de hdrflags (Etapa 1).'),
    'fe_mqtt_qos': ('CONSERVA', 'QoS decodificado de hdrflags (Etapa 1).'),
    'fe_mqtt_dup': ('CONSERVA', 'Flag DUP decodificado de hdrflags (Etapa 1).'),
    'fe_mqtt_retain': ('CONSERVA', 'Flag RETAIN decodificado de hdrflags (Etapa 1).'),
    'fe_topic_wildcard': ('CONSERVA', 'Topic con #, + o $SYS (Etapa 1).'),
    'fe_topic_novedad': ('CONSERVA', 'Topic no observado en train (Etapa 2).'),
    'fe_has_mqtt': ('CONSERVA', '1 si mqtt.hdrflags no esta vacio; 0 si no (Etapa 1).'),
    'fe_cap_ratio': ('DESCARTA', 'Constante 1.0 con frame.cap_len igual a frame.len.'),
    'fe_anon_connect': ('DESCARTA', 'Casi siempre 0; no separa normal de ataque.'),
}
# Motivos de descarte que no dependen de la tasa de nulos.
RAZONES = {
    'frame.time_delta_displayed': 'Duplica frame.time_delta',
    'frame.time_epoch': 'Fuga de captura (reloj absoluto)',
    'frame.time_relative': 'Fuga de captura (reloj relativo)',
    'frame.number': 'Fuga de captura (orden de trama)',
    'frame.cap_len': 'Duplica frame.len',
    'frame.encap_type': 'Constante',
    'frame.ignored': 'Constante',
    'frame.marked': 'Constante',
    'frame.offset_shift': 'Constante',
    'ip.src': 'Identidad de host',
    'ip.dst': 'Identidad de host',
    'eth.src': 'Identidad de capa 2',
    'eth.dst': 'Identidad de capa 2',
    'mqtt.clientid': 'Identidad de cliente',
    'mqtt.msgtype': 'Derivable de mqtt.hdrflags',
    'mqtt.qos': 'Derivable de mqtt.hdrflags',
    'mqtt.retain': 'Derivable de mqtt.hdrflags',
    'mqtt.dupflag': 'Derivable de mqtt.hdrflags',
    'mqtt.msg': 'Texto crudo (se resume en entropia)',
    'mqtt.topic': 'Texto crudo (se resume en entropia)',
    'tcp.srcport': 'Puerto efimero (solo bandera)',
    'tcp.dstport': 'Puerto efimero (solo bandera)',
}
# Columnas internas que no son features.
INTERNAS = ('type', '_capture', '_original_row', '_observation', '_endpoints', '_connection', '_direction')



### Catálogos de contexto y constantes de ventana

Declara las tuplas de CONNECT, de contexto por origen y de contexto L2, las
ventanas temporales de 5 s, la MAC de broadcast y el umbral de ráfaga. Solo
definiciones.


In [6]:
# Contexto de CONNECT: (feature, columna de transporte, columna cruda de origen).
CONNECT_CONTEXT = (
    ('fe_mqtt_ver', '_connect_ver', 'mqtt.ver'),
    ('fe_mqtt_kalive', '_connect_kalive', 'mqtt.kalive'),
    ('fe_mqtt_cleansess', '_connect_cleansess', 'mqtt.conflag.cleansess'),
    ('fe_mqtt_clientid_len', '_connect_clientid_len', 'mqtt.clientid_len'),
)
# Contexto por origen: (feature, columna de transporte).
ORIGIN_CONTEXT = [
    ('fe_origen_frames_s', '_ctx_origen_frames_s'),
    ('fe_origen_bytes_s', '_ctx_origen_bytes_s'),
    ('fe_origen_conexiones', '_ctx_origen_conexiones'),
    ('fe_origen_connects', '_ctx_origen_connects'),
]
ORIGIN_CONTEXT_WINDOW_SECONDS = 5.0
# Contexto L2 por eth.src: (feature, columna de transporte).
L2_CONTEXT = [
    ('fe_is_broadcast', '_ctx_is_broadcast'),
    ('fe_is_multicast', '_ctx_is_multicast'),
    ('fe_l2_frames_s', '_ctx_l2_frames_s'),
    ('fe_l2_broadcasts', '_ctx_l2_broadcasts'),
    ('fe_l2_burst_ratio', '_ctx_l2_burst_ratio'),
]
L2_CONTEXT_WINDOW_SECONDS = 5.0
BROADCAST_MAC = 'ff:ff:ff:ff:ff:ff'
BURST_DELTA_SECONDS = 100e-6


### Orden final de columnas y tratamiento de NaN

Fija `BASE_FEATURE_COLUMNS` (Etapa 1), `NAN_FEATURES` y `FEATURE_COLUMNS`
(Etapa 1 + Etapa 2). El orden lo consumen el modelo y los adapters. Solo
definiciones.


In [7]:
# Orden final de la matriz del modelo.
BASE_FEATURE_COLUMNS = [
    'frame.len', 'frame.time_delta',
    'fe_bytes_per_sec', 'fe_is_standard_mqtt_port', 'fe_is_broker_to_client', 'fe_log_len',
    'mqtt.len', 'mqtt.topic_len',
    'fe_mqtt_type', 'fe_mqtt_qos', 'fe_mqtt_dup', 'fe_mqtt_retain',
    'fe_msg_entropy', 'fe_topic_entropy', 'fe_topic_depth', 'fe_payload_ratio',
    'fe_mqtt_ver', 'fe_mqtt_kalive', 'fe_mqtt_cleansess', 'fe_mqtt_clientid_len',
    'fe_topic_wildcard', 'fe_has_mqtt',
]
# Features no observables que conservan NaN (XGBoost) o 0 post-escalado (LSTM).
NAN_FEATURES = [
    'mqtt.len', 'mqtt.topic_len',
    'fe_mqtt_type', 'fe_mqtt_qos', 'fe_mqtt_dup', 'fe_mqtt_retain',
    'fe_msg_entropy', 'fe_topic_entropy', 'fe_topic_depth', 'fe_payload_ratio',
    'fe_mqtt_ver', 'fe_mqtt_kalive', 'fe_mqtt_cleansess', 'fe_mqtt_clientid_len',
    'fe_topic_wildcard', 'fe_topic_novedad',
    'fe_is_broadcast', 'fe_is_multicast',
    'fe_origen_frames_s', 'fe_origen_bytes_s', 'fe_origen_conexiones',
    'fe_origen_connects',
    'fe_l2_frames_s', 'fe_l2_broadcasts', 'fe_l2_burst_ratio',
]
FEATURE_COLUMNS = (BASE_FEATURE_COLUMNS + ['fe_origen_conocido', 'fe_origen_observable']
                   + [feature for feature, _ in ORIGIN_CONTEXT]
                   + [feature for feature, _ in L2_CONTEXT]
                   + ['fe_topic_novedad'])

### Diagnóstico de columnas crudas

Define `diagnostico_columnas` e `imprimir_tabla` y, al ejecutar, imprime el
diagnóstico de columnas y el resumen de `hdrflags`. **Ejecutar; no entrena.**


In [8]:
def diagnostico_columnas(df):
    """Clasifica cada columna cruda: nulos, cardinalidad, decision y motivo."""
    filas = []
    for col in df.columns:
        if col in INTERNAS:
            continue
        nulos = float(df[col].isna().mean())
        distintos = int(df[col].nunique(dropna=True))
        if col in KEEP_RAW:
            decision, motivo = 'CONSERVA', 'Feature cruda con senal'
        elif col in DERIVED_SOURCES:
            decision, motivo = 'DERIVA', 'Resume en ' + DERIVED_SOURCES[col]
        elif col in RAZONES:
            decision, motivo = 'DESCARTA', RAZONES[col]
        elif nulos == 1.0:
            decision, motivo = 'DESCARTA', 'Vacia (100% nulos)'
        elif nulos >= 0.99:
            decision, motivo = 'DESCARTA', 'Metadato esporadico (>99% nula)'
        else:
            decision, motivo = 'DESCARTA', 'Sin senal'
        filas.append({'columna': col, 'nulos%': round(100.0 * nulos, 2),
                      'distintos': distintos, 'decision': decision, 'motivo': motivo})
    return pd.DataFrame(filas)


def imprimir_tabla(tabla, columnas):
    """Imprime un DataFrame en columnas de ancho fijo alineadas."""
    anchos = {c: max(len(c), max(len(str(v)) for v in tabla[c])) for c in columnas}
    print('  '.join(c.ljust(anchos[c]) for c in columnas))
    print('  '.join('-' * anchos[c] for c in columnas))
    for _, fila in tabla.iterrows():
        print('  '.join(str(fila[c]).ljust(anchos[c]) for c in columnas))


diagnostico = diagnostico_columnas(df_raw)
orden = {'CONSERVA': 0, 'DERIVA': 1, 'DESCARTA': 2}
diagnostico['_orden'] = diagnostico['decision'].map(orden)
diagnostico = (diagnostico.sort_values(['_orden', 'nulos%'], ascending=[True, False])
                          .drop(columns='_orden').reset_index(drop=True))
conteo = diagnostico['decision'].value_counts()
print('DIAGNOSTICO DE COLUMNAS CRUDAS')
print(f"evaluadas={len(diagnostico)}  conserva={conteo.get('CONSERVA', 0)}  "
      f"deriva={conteo.get('DERIVA', 0)}  descarta={conteo.get('DESCARTA', 0)}")
print()
imprimir_tabla(diagnostico, ['columna', 'nulos%', 'distintos', 'decision', 'motivo'])
print()
print('FEATURES DERIVADAS')
derivadas = pd.DataFrame([{'derivada': nombre, 'decision': datos[0], 'motivo': datos[1]}
                          for nombre, datos in DERIVED_FEATURES.items()])
imprimir_tabla(derivadas, ['derivada', 'decision', 'motivo'])
hdr = df_raw['mqtt.hdrflags']
hdr_present = hdr.notna() & hdr.astype('string').str.strip().ne('')
hdr_multi = int((hdr_present & hdr.astype('string').str.contains(',')).sum())
print()
print(f'hdrflags: presentes={int(hdr_present.sum())} multi_valor={hdr_multi}')

DIAGNOSTICO DE COLUMNAS CRUDAS
evaluadas=66  conserva=5  deriva=4  descarta=57

columna                     nulos%  distintos  decision  motivo                                     
--------------------------  ------  ---------  --------  -------------------------------------------
mqtt.topic_len              85.13   61         CONSERVA  Feature cruda con senal                    
mqtt.hdrflags               83.24   35         CONSERVA  Feature cruda con senal                    
mqtt.len                    83.24   81         CONSERVA  Feature cruda con senal                    
frame.time_delta            0.0     42651      CONSERVA  Feature cruda con senal                    
frame.len                   0.0     1462       CONSERVA  Feature cruda con senal                    
mqtt.msg                    85.4    6056       DERIVA    Resume en fe_msg_entropy                   
mqtt.topic                  85.39   5905       DERIVA    Resume en fe_topic_entropy y fe_topic_depth
tcp.srcport

### 2.1 Constantes y decodificación de `hdrflags`

Define `NETWORK_RAW`, las columnas numéricas MQTT, el máximo de un byte y
`decode_hdrflags`, que extrae tipo, QoS, DUP y RETAIN del primer byte de
`mqtt.hdrflags` y devuelve `NaN` fuera de rango. Solo define.


In [9]:
NETWORK_RAW = ['frame.len', 'frame.time_delta']
MQTT_NUMERIC_COLUMNS = ['mqtt.len', 'mqtt.topic_len']


HDRFLAGS_MAX = 0xFF

def decode_hdrflags(value):
    """Decodifica el primer byte MQTT: tipo, QoS, DUP y RETAIN.

    Usa la primera ocurrencia separada por comas (igual que la captura) y
    rechaza valores fuera de un byte en vez de truncar bits.
    """
    if pd.isna(value) or not str(value).strip():
        return (np.nan, np.nan, np.nan, np.nan)
    text = str(value).strip().split(',')[0].strip()
    try:
        header = int(text, 16) if text.lower().startswith('0x') else int(float(text))
    except ValueError:
        return (np.nan, np.nan, np.nan, np.nan)
    if not 0 <= header <= HDRFLAGS_MAX:
        return (np.nan, np.nan, np.nan, np.nan)
    return ((header >> 4) & 0x0F, (header >> 1) & 0x03,
            (header >> 3) & 0x01, header & 0x01)




### 2.2 `topic_wildcard`

`topic_wildcard` marca con 1 los topics que contienen `#`, `+` o comienzan con
`$SYS`; devuelve `NaN` si no hay topic. Solo define.


In [10]:
def topic_wildcard(value):
    if pd.isna(value) or not str(value):
        return np.nan
    text = str(value)
    return 1 if ('#' in text or '+' in text or text.startswith('$SYS')) else 0




### 2.3 Features de red (`build_network_features`)

Toma `frame.len` y `frame.time_delta` como base, añade `fe_bytes_per_sec` (desde
la tasa causal `_ctx_bytes_per_s`), las banderas de puerto MQTT y `fe_log_len`.
**Usa el estado causal de tasa de bytes**; no resume identidades. Solo define.


In [11]:
def build_network_features(X_full):
    """Red/transporte sin puertos crudos ni reloj de captura."""
    base = X_full[NETWORK_RAW].apply(pd.to_numeric, errors='coerce').fillna(NAN_FILL).copy()
    rate = (pd.to_numeric(X_full['_ctx_bytes_per_s'], errors='coerce').fillna(0.0)
            if '_ctx_bytes_per_s' in X_full
            else pd.Series(0.0, index=X_full.index))
    base['fe_bytes_per_sec'] = np.log1p(rate.clip(lower=0.0))
    dst = pd.to_numeric(X_full['tcp.dstport'], errors='coerce')
    src = pd.to_numeric(X_full['tcp.srcport'], errors='coerce')
    base['fe_is_standard_mqtt_port'] = dst.isin([1883, 8883]).astype(int)
    base['fe_is_broker_to_client'] = src.isin([1883, 8883]).astype(int)
    base['fe_log_len'] = np.log1p(base['frame.len'])
    return base




### 2.4 Contexto CONNECT (`add_connect_context`)

Propaga `mqtt.ver`, `mqtt.kalive`, `mqtt.conflag.cleansess` y `mqtt.clientid_len`
hacia adelante dentro de cada `_connection`. Trabaja por grupo y no reordena las
filas. Solo define.


In [12]:
def add_connect_context(df):
    """Forward-fill causal del CONNECT dentro de cada _connection (Etapa 1)."""
    result = df.copy()
    grouped = result.groupby('_connection', sort=False)
    for feature, transport, source in CONNECT_CONTEXT:
        if source in result:
            result[transport] = grouped[source].ffill()
        else:
            result[transport] = np.nan
    return result




### 2.5 Tasa causal de bytes (`build_direction_bytes_rate`)

Recorre cada partición en orden de captura y tiempo, y escribe `_ctx_bytes_per_s`:
suma `frame.len` de frames anteriores de la misma **dirección** en `[t-5, t]`,
excluye el frame actual y divide entre 5. Sin historial vale 0. El estado se
reinicia por captura y retroceso de reloj. **La clave es dirección.**


In [13]:
def build_direction_bytes_rate(df, partitions, window_seconds=DIRECTION_RATE_WINDOW_SECONDS):
    """Tasa causal de bytes por conexion/episodio y direccion (Fase 4.3).

    Suma `frame.len` de frames anteriores de la misma `_direction` con tiempo en
    `[t-5, t]`, excluye el frame actual y divide entre 5. Sin historial -> 0.
    Longitudes faltantes no aportan bytes inventados.
    """
    epochs = pd.to_numeric(df['frame.time_epoch'], errors='coerce').to_numpy()
    lengths = pd.to_numeric(df['frame.len'], errors='coerce').to_numpy()
    directions = df['_direction'].to_numpy()
    captures = df['_capture'].astype('string').to_numpy()
    values = np.zeros(len(df), dtype=float)
    invalid_lengths = 0
    for rows in partitions.values():
        state = {}
        selected = np.asarray(rows, dtype=int)
        order = np.lexsort((epochs[selected], captures[selected]))
        current_capture = None
        for row in selected[order]:
            if captures[row] != current_capture:
                state = {}
                current_capture = captures[row]
            key = directions[row]
            epoch = epochs[row]
            length = lengths[row]
            if key is None or not np.isfinite(epoch):
                continue
            events = state.setdefault(key, [])
            if events and epoch < events[-1]['epoch']:
                events.clear()
            while events and epoch - events[0]['epoch'] > window_seconds:
                events.pop(0)
            values[row] = sum(event['len'] for event in events) / window_seconds
            if np.isfinite(length):
                events.append({'epoch': float(epoch), 'len': float(length)})
            else:
                invalid_lengths += 1
    df['_ctx_bytes_per_s'] = values
    print(f'  tasa de bytes: longitudes invalidas excluidas={invalid_lengths}')
    return df




### 2.6 Contexto por origen (`build_origin_context`)

Ventana causal de 5 s por `ip.src` sobre frames anteriores; escribe las cuatro
columnas `_ctx_origen_*` (frames/s, bytes/s, conexiones y CONNECTs). El estado se
reinicia al inicio de cada partición y captura. **Trabaja por IP de origen.**


In [14]:
def build_origin_context(df, partitions, window_seconds=ORIGIN_CONTEXT_WINDOW_SECONDS):
    """Ventana causal de 5 s por ip.src; reset al inicio de cada particion."""
    epochs = pd.to_numeric(df['frame.time_epoch'], errors='coerce').to_numpy()
    lengths = pd.to_numeric(df['frame.len'], errors='coerce').to_numpy()
    ips = df['ip.src'].astype('string').str.strip().to_numpy()
    connections = df['_connection'].to_numpy()
    msgtypes = pd.to_numeric(df['mqtt.msgtype'], errors='coerce').to_numpy()
    captures = df['_capture'].astype('string').to_numpy()
    values = {transport: np.full(len(df), np.nan) for _, transport in ORIGIN_CONTEXT}
    for rows in partitions.values():
        state = {}
        selected = np.asarray(rows, dtype=int)
        order = np.lexsort((epochs[selected], captures[selected]))
        current_capture = None
        for row in selected[order]:
            if captures[row] != current_capture:
                state = {}
                current_capture = captures[row]
            ip = ips[row]
            epoch = epochs[row]
            if pd.isna(ip) or ip == '' or not np.isfinite(epoch):
                continue
            events = state.setdefault(ip, [])
            if events and epoch < events[-1]['epoch']:
                events.clear()
            while events and epoch - events[0]['epoch'] > window_seconds:
                events.pop(0)
            if events:
                values['_ctx_origen_frames_s'][row] = len(events) / window_seconds
                values['_ctx_origen_bytes_s'][row] = (
                    sum(event['len'] for event in events) / window_seconds)
                values['_ctx_origen_conexiones'][row] = len({event['conn'] for event in events})
                values['_ctx_origen_connects'][row] = sum(event['connect'] for event in events)
            else:
                for _, transport in ORIGIN_CONTEXT:
                    values[transport][row] = 0.0
            events.append({
                'epoch': float(epoch),
                'len': float(lengths[row]) if np.isfinite(lengths[row]) else 0.0,
                'conn': connections[row],
                'connect': 1 if msgtypes[row] == 1 else 0,
            })
    for _, transport in ORIGIN_CONTEXT:
        df[transport] = values[transport]
    return df




### 2.7 Contexto L2 (`_multicast_flag` y `build_l2_context`)

`_multicast_flag` lee el bit multicast del primer octeto de una MAC. A partir de
`eth.dst` se calculan `fe_is_broadcast` y `fe_is_multicast`; la ventana causal de
5 s se agrupa por **`eth.src`** y produce `fe_l2_frames_s`, `fe_l2_broadcasts` y
`fe_l2_burst_ratio`. No mezcla `eth.src` con `eth.dst`.


In [15]:
def _multicast_flag(value):
    if pd.isna(value) or not str(value).strip():
        return np.nan
    first = str(value).strip().lower().split(':')
    try:
        return int(int(first[0], 16) & 1)
    except (ValueError, IndexError):
        return np.nan


def build_l2_context(df, partitions, window_seconds=L2_CONTEXT_WINDOW_SECONDS):
    """Ventana causal de 5 s por eth.src; sin mezclar eth.src y eth.dst."""
    epochs = pd.to_numeric(df['frame.time_epoch'], errors='coerce').to_numpy()
    sources = df['eth.src'].astype('string').str.strip().to_numpy()
    dests = df['eth.dst'].astype('string').str.strip()
    dests_lower = dests.str.lower()
    valid_dest = (dests_lower.notna() & dests_lower.ne('')).fillna(False)
    # Destino ausente o invalido -> indicadores NaN, no unicast valido.
    broadcast = (dests_lower.eq(BROADCAST_MAC).fillna(False).astype(float)
                 .where(valid_dest).to_numpy())
    multicast = dests_lower.map(_multicast_flag).where(valid_dest).to_numpy()
    captures = df['_capture'].astype('string').to_numpy()
    values = {transport: np.full(len(df), np.nan) for _, transport in L2_CONTEXT}
    values['_ctx_is_broadcast'][:] = broadcast.astype(float)
    values['_ctx_is_multicast'][:] = multicast.astype(float)
    for rows in partitions.values():
        state = {}
        selected = np.asarray(rows, dtype=int)
        order = np.lexsort((epochs[selected], captures[selected]))
        current_capture = None
        for row in selected[order]:
            if captures[row] != current_capture:
                state = {}
                current_capture = captures[row]
            source = sources[row]
            epoch = epochs[row]
            values['_ctx_is_broadcast'][row] = float(broadcast[row])
            values['_ctx_is_multicast'][row] = float(multicast[row])
            if pd.isna(source) or source == '' or not np.isfinite(epoch):
                continue
            events = state.setdefault(source, [])
            if events and epoch < events[-1]['epoch']:
                events.clear()
            while events and epoch - events[0]['epoch'] > window_seconds:
                events.pop(0)
            if events:
                values['_ctx_l2_frames_s'][row] = len(events) / window_seconds
                values['_ctx_l2_broadcasts'][row] = sum(e['broadcast'] for e in events)
                deltas = [events[i]['epoch'] - events[i - 1]['epoch']
                          for i in range(1, len(events))]
                values['_ctx_l2_burst_ratio'][row] = (
                    sum(delta < BURST_DELTA_SECONDS for delta in deltas) / len(deltas)
                    if deltas else 0.0)
            else:
                values['_ctx_l2_frames_s'][row] = 0.0
                values['_ctx_l2_broadcasts'][row] = 0.0
                values['_ctx_l2_burst_ratio'][row] = 0.0
            events.append({'epoch': float(epoch),
                           'broadcast': 0.0 if pd.isna(broadcast[row]) else float(broadcast[row])})
    for _, transport in L2_CONTEXT:
        df[transport] = values[transport]
    return df




### 2.8 Matriz MQTT (`build_mqtt_features` y `build_feature_matrix`)

`build_mqtt_features` compone la matriz compacta: red, columnas MQTT numéricas,
derivadas de `hdrflags`, entropía, profundidad y ratio; rellena los NaN no
faltantes y recorta a `BASE_FEATURE_COLUMNS`. `build_feature_matrix` es el atajo
de Etapa 1. Solo definen.


In [16]:
def build_mqtt_features(df):
    """Matriz compacta MQTT: sin identidades, puertos crudos ni reloj de captura."""
    result = build_network_features(df)

    def number(value):
        if isinstance(value, str) and value.lower().startswith('0x'):
            try:
                return int(value, 16)
            except ValueError:
                return np.nan
        return value

    for col in MQTT_NUMERIC_COLUMNS:
        series = df[col] if col in df else pd.Series(np.nan, index=df.index)
        result[col] = pd.to_numeric(series.map(number), errors='coerce')
    msg = df['mqtt.msg'] if 'mqtt.msg' in df else pd.Series(None, index=df.index, dtype=object)
    topic = df['mqtt.topic'] if 'mqtt.topic' in df else pd.Series(None, index=df.index, dtype=object)
    hdr = df['mqtt.hdrflags'] if 'mqtt.hdrflags' in df else pd.Series(None, index=df.index, dtype=object)
    decoded = hdr.apply(decode_hdrflags)
    result['fe_mqtt_type'] = [value[0] for value in decoded]
    result['fe_mqtt_qos'] = [value[1] for value in decoded]
    result['fe_mqtt_dup'] = [value[2] for value in decoded]
    result['fe_mqtt_retain'] = [value[3] for value in decoded]
    result['fe_topic_wildcard'] = topic.apply(topic_wildcard)
    result['fe_has_mqtt'] = hdr.apply(
        lambda value: 0 if pd.isna(value) or not str(value).strip() else 1)
    result['fe_msg_entropy'] = msg.apply(entropia)
    result['fe_topic_entropy'] = topic.apply(entropia)
    result['fe_topic_depth'] = topic.apply(
        lambda value: np.nan if pd.isna(value) or not str(value) else str(value).count('/') + 1)
    result['fe_payload_ratio'] = result['mqtt.len'].clip(lower=0) / (
        result['frame.len'].clip(lower=0) + 1e-5)
    for feature, transport, source in CONNECT_CONTEXT:
        series = df[transport] if transport in df else pd.Series(np.nan, index=df.index)
        result[feature] = pd.to_numeric(series, errors='coerce')
    result = result.replace([np.inf, -np.inf], np.nan)
    fill_columns = [column for column in result.columns if column not in NAN_FEATURES]
    result[fill_columns] = result[fill_columns].fillna(NAN_FILL)
    return result


def build_feature_matrix(df):
    """Matriz de Etapa 1 en el orden de BASE_FEATURE_COLUMNS."""
    return build_mqtt_features(df)[BASE_FEATURE_COLUMNS]

### 2.9 Matriz de Etapa 1 y etiquetas

Ejecuta `add_connect_context` y `build_feature_matrix`, ajusta el `LabelEncoder`
de las cuatro clases y valida tipos y finitud. **Al ejecutar esta celda se
construyen `X_stage1` y `y`; no entrena modelos.**


In [17]:
CLASS_NAMES = ['normal', 'DoS', 'mitm', 'intrusion']
df_raw = add_connect_context(df_raw)
X_stage1 = build_feature_matrix(df_raw)
le_target = LabelEncoder().fit(CLASS_NAMES)
y = pd.Series(le_target.transform(df_raw[TARGET_COL]), index=df_raw.index)
normal_idx = int(le_target.transform(['normal'])[0])
assert all(pd.api.types.is_numeric_dtype(X_stage1[col]) for col in X_stage1)
finite_stage1 = X_stage1.to_numpy(dtype=float)
assert np.isfinite(finite_stage1[~np.isnan(finite_stage1)]).all()
print(f'Matriz Etapa 1: {X_stage1.shape}; clases: {list(CLASS_NAMES)}')
print('Features Etapa 1:', list(X_stage1.columns))
hdr_present = df_raw['mqtt.hdrflags'].notna() & df_raw['mqtt.hdrflags'].astype('string').str.strip().ne('')
decoded_hdr = df_raw['mqtt.hdrflags'].apply(decode_hdrflags)
hdr_invalid = int((hdr_present & decoded_hdr.apply(
    lambda parts: all(pd.isna(part) for part in parts))).sum())
print(f'hdrflags invalidos (fuera de byte o no parseables): {hdr_invalid}')

Matriz Etapa 1: (286186, 22); clases: ['normal', 'DoS', 'mitm', 'intrusion']
Features Etapa 1: ['frame.len', 'frame.time_delta', 'fe_bytes_per_sec', 'fe_is_standard_mqtt_port', 'fe_is_broker_to_client', 'fe_log_len', 'mqtt.len', 'mqtt.topic_len', 'fe_mqtt_type', 'fe_mqtt_qos', 'fe_mqtt_dup', 'fe_mqtt_retain', 'fe_msg_entropy', 'fe_topic_entropy', 'fe_topic_depth', 'fe_payload_ratio', 'fe_mqtt_ver', 'fe_mqtt_kalive', 'fe_mqtt_cleansess', 'fe_mqtt_clientid_len', 'fe_topic_wildcard', 'fe_has_mqtt']
hdrflags invalidos (fuera de byte o no parseables): 0


<a id="sec-particiones"></a>

## 3. Particiones agrupadas y temporales (Fase 5)

Se conserva `StratifiedGroupKFold` (5 folds, semilla 42) con conexiones/episodios
completos como grupos. La rotación usa `test=fold k`, `validation=fold (k+1) % 5`
y `train` los tres restantes; no se busca otra combinación. El split temporal
corta por percentiles 60/80 dentro de cada captura y descarta (contándolos) los
grupos que atraviesan un corte.


### 3.1 Identificadores, ventanas y folds

Traduce `_connection` y `_direction` a enteros, define las ventanas por dirección
(buffer inicial de 11) y los folds agrupados con la rotación fija test=`k`,
validation=`(k+1) % 5`. Al ejecutar crea `FOLDS`; no entrena.


In [18]:
connection_ids = pd.factorize(df_raw['_connection'], sort=False)[0]
direction_ids = pd.factorize(df_raw['_direction'], sort=False)[0]


def sequence_indices(indices, seq_length=SEQ_LEN):
    """Ventanas por dirección TCP/L2 y partición; buffer inicial de 11."""
    selected = np.asarray(indices, dtype=int)
    windows = []
    grouped = pd.Series(direction_ids[selected], index=selected)
    for members in grouped.groupby(grouped, sort=False).groups.values():
        rows = np.asarray(members, dtype=int)
        rows = rows[np.argsort(df_raw.iloc[rows]['frame.time_epoch'].to_numpy(), kind='stable')]
        for start in range(1, len(rows) - seq_length + 1):
            windows.append(rows[start:start + seq_length])
    return np.asarray(windows, dtype=int).reshape(-1, seq_length)


def grouped_folds(seed=SEED, n_splits=N_SPLITS):
    """Folds agrupados por conexión/episodio, semilla y tamaño fijos (5.1)."""
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [test for _, test in splitter.split(X_stage1, y, groups=connection_ids)]


def fold_partitions(folds, k):
    """Rotación fija test=k, validation=(k+1) % n, train=resto (5.1)."""
    n = len(folds)
    validation = (k + 1) % n
    return {
        'train': np.concatenate([folds[i] for i in range(n)
                                 if i not in (k, validation)]),
        'val': folds[validation],
        'test': folds[k],
    }


def class_counts(rows):
    """Frames y grupos independientes por clase para cobertura (5.3/5.5)."""
    labels = y.iloc[rows].to_numpy()
    groups = connection_ids[rows]
    return {
        name: {'frames': int((labels == index).sum()),
               'groups': int(len(set(groups[labels == index])))}
        for index, name in enumerate(CLASS_NAMES)
    }


def temporal_partitions(frame, groups, train_quantile=0.60, val_quantile=0.80):
    """Split cronológico por captura con grupos completos (5.4)."""
    epochs = pd.to_numeric(frame['frame.time_epoch'], errors='coerce').to_numpy()
    captures = frame['_capture'].astype('string').to_numpy()
    buckets = {'train': [], 'val': [], 'test': []}
    excluded = []
    for capture in pd.unique(captures):
        mask = captures == capture
        cut_train = float(np.percentile(epochs[mask], 100 * train_quantile))
        cut_val = float(np.percentile(epochs[mask], 100 * val_quantile))
        for group in pd.unique(groups[mask]):
            member = groups == group
            start = float(np.nanmin(epochs[member]))
            end = float(np.nanmax(epochs[member]))
            if end <= cut_train:
                buckets['train'].append(group)
            elif start >= cut_train and end <= cut_val:
                buckets['val'].append(group)
            elif start >= cut_val:
                buckets['test'].append(group)
            else:
                excluded.append(group)
    partitions = {name: np.flatnonzero(np.isin(groups, values))
                  for name, values in buckets.items()}
    return partitions, excluded


FOLDS = grouped_folds()
print('Folds agrupados (frames):', [int(len(fold)) for fold in FOLDS])


Folds agrupados (frames): [67950, 44382, 50687, 66408, 56759]


<a id="sec-estado"></a>

## 4. Estado de features ajustado por partición

Cada ajuste (fold o split temporal) reconstruye las allowlists, el contexto
causal, la imputación y el `StandardScaler` usando **solo** su train. El costo de
MSE del híbrido se obtiene out-of-fold dentro del train externo (5.2).


### 4.1 Ajuste por partición (`fit_feature_state`)

Define el ajuste de allowlists, contexto causal, imputación y `StandardScaler`
usando **solo** las filas de train de la partición. Devuelve el estado completo
que consumen los modelos. **Solo define; no entrena.**


In [19]:
def _clean_identity_values(series):
    values = series.dropna().astype(str).str.strip()
    return {value for value in values if value}


def fit_feature_state(frame, partitions, train_idx, include_origin=True):
    """Features, allowlists, contexto, imputación y scaler ajustados solo en train."""
    work = frame.copy()
    train_frame = work.iloc[train_idx]
    normal_train = train_frame[train_frame[TARGET_COL].eq('normal')]
    known_ips = _clean_identity_values(normal_train['ip.src'])
    known_clientids = _clean_identity_values(normal_train['mqtt.clientid'])
    known_origins = ({'ip:' + value for value in known_ips}
                     | {'client:' + value for value in known_clientids})
    train_topics = train_frame['mqtt.topic'].astype('string').str.strip()
    known_topics = {value for value in train_topics.dropna() if value}

    work = build_direction_bytes_rate(work, partitions)
    work = build_origin_context(work, partitions)
    work = build_l2_context(work, partitions)
    X_stage1 = build_feature_matrix(work)

    all_ips = work['ip.src'].astype('string').str.strip()
    all_clients = work['mqtt.clientid'].astype('string').str.strip()
    X = X_stage1.copy()
    X['fe_origen_conocido'] = (
        all_ips.isin(known_ips) | all_clients.isin(known_clientids)
    ).to_numpy(dtype=bool).astype(int)
    ip_observable = all_ips.notna() & all_ips.ne('')
    client_observable = all_clients.notna() & all_clients.ne('')
    X['fe_origen_observable'] = (
        ip_observable | client_observable
    ).fillna(False).to_numpy(dtype=bool).astype(int)
    for feature, transport in ORIGIN_CONTEXT:
        X[feature] = work[transport].to_numpy()
    for feature, transport in L2_CONTEXT:
        X[feature] = work[transport].to_numpy()
    all_topics = work['mqtt.topic'].astype('string').str.strip()
    topic_present = all_topics.notna() & all_topics.ne('')
    X['fe_topic_novedad'] = np.where(
        topic_present, (~all_topics.isin(known_topics)).astype(float), np.nan)
    X = X[FEATURE_COLUMNS]
    if not include_origin:
        X = X.drop(columns=['fe_origen_conocido', 'fe_origen_observable'])
    finite = X.to_numpy(dtype=float)
    assert np.isfinite(finite[~np.isnan(finite)]).all()

    base_columns = list(X.columns)
    raw_train = X.iloc[train_idx]
    all_missing_in_train = [column for column in base_columns
                            if raw_train[column].isna().all()]
    train_means = raw_train.mean(skipna=True)
    X_filled = X.fillna(train_means.fillna(0.0))
    scaler = StandardScaler().fit(X_filled.iloc[train_idx])
    X_scaled = scaler.transform(X_filled).astype(np.float32)
    X_scaled[X.isna().to_numpy()] = 0.0
    missing_mask_columns = ['missing__' + column for column in NAN_FEATURES]
    missing_masks = X[NAN_FEATURES].isna().to_numpy(dtype=np.float32)
    tensor_columns = base_columns + missing_mask_columns
    X_tensor = np.concatenate([X_scaled, missing_masks], axis=1).astype(np.float32)
    assert np.isfinite(X_tensor).all()
    assert X_tensor.shape[1] == len(tensor_columns)
    return {
        'work': work, 'X': X, 'X_stage1': X_stage1,
        'X_scaled': X_scaled, 'X_tensor': X_tensor,
        'known_origins': known_origins, 'known_topics': known_topics,
        'base_columns': base_columns, 'tensor_columns': tensor_columns,
        'missing_mask_columns': missing_mask_columns,
        'scaler': scaler, 'train_means': train_means,
        'all_missing_in_train': all_missing_in_train,
    }


<a id="sec-modelos"></a>

## 5. Modelos, entrenamiento y helpers

El LSTM público es supervisado (multiclase); el autoencoder auxiliar reconstruye
el tensor documentado. Las variantes A/B/C de 5.5 solo cambian máscaras y
proyección del decoder.


### 5.1 Modelos base

Definen XGBoost multiclase con `multi:softprob` y las arquitecturas LSTM: el
clasificador supervisado `LSTMSupervised` y los dos decoders del autoencoder
(recurrente y proyectado). Solo definen clases y constructores.


In [20]:
def get_xgb_model():
    return XGBClassifier(objective='multi:softprob', num_class=len(CLASS_NAMES),
                         n_estimators=100, learning_rate=0.1, max_depth=6,
                         tree_method='hist', device=XGB_DEVICE, random_state=SEED,
                         eval_metric='mlogloss')


def fit_classifier(features, labels):
    """Ajusta XGBoost multiclase con pesos de las cuatro clases de train."""
    model = get_xgb_model()
    model.fit(features, labels,
              sample_weight=compute_sample_weight('balanced', np.asarray(labels)))
    return model


class LSTMSupervised(nn.Module):
    """Encoder LSTM (hidden 32) -> Linear(32, 4) sobre el ultimo estado oculto."""
    def __init__(self, num_features, hidden_dim=32, num_classes=len(CLASS_NAMES)):
        super().__init__()
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        return self.classifier(hidden[-1])


class LSTMAutoencoderRecurrent(nn.Module):
    """Decoder recurrente que produce directamente las features (variantes A/B)."""
    def __init__(self, seq_len, num_features, hidden_dim=32):
        super().__init__()
        self.seq_len = seq_len
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_dim, num_features, batch_first=True)

    def forward(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        latent = hidden.permute(1, 0, 2).repeat(1, self.seq_len, 1)
        output, _ = self.decoder_lstm(latent)
        return output


class LSTMAutoencoderProjected(nn.Module):
    """Decoder recurrente + proyeccion lineal a las features (variante C)."""
    def __init__(self, seq_len, num_features, hidden_dim=32):
        super().__init__()
        self.seq_len = seq_len
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.projection = nn.Linear(hidden_dim, num_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        latent = hidden.permute(1, 0, 2).repeat(1, self.seq_len, 1)
        output, _ = self.decoder_lstm(latent)
        return self.projection(output)




### 5.2 Pesos y entrenamiento

`class_weights` balancea las cuatro clases. `train_supervised_lstm` entrena el
LSTM multiclase con entropía cruzada ponderada; `train_autoencoder` entrena el
autoencoder auxiliar solo con ventanas completamente normales. Solo definen.


In [21]:
def class_weights(labels):
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=len(CLASS_NAMES))
    weights = len(labels) / (len(CLASS_NAMES) * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)


def train_supervised_lstm(sequences, num_features):
    """LSTM supervisado multiclase; target = clase de la ultima fila."""
    xs, labels = sequences['train'][0], sequences['train'][1]
    model = LSTMSupervised(num_features).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(labels).to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loader = DataLoader(TensorDataset(torch.from_numpy(xs),
                                      torch.from_numpy(labels).long()),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    model.train()
    for _ in range(EPOCHS):
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
    return model


def train_autoencoder(train_x, projection):
    """Autoencoder auxiliar entrenado solo con ventanas completamente normales."""
    if not len(train_x):
        raise ValueError('No hay ventanas completamente normales de entrenamiento')
    architecture = LSTMAutoencoderProjected if projection else LSTMAutoencoderRecurrent
    model = architecture(SEQ_LEN, train_x.shape[2]).to(DEVICE)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_x)),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for _ in range(EPOCHS):
        for (inputs,) in loader:
            inputs = inputs.to(DEVICE)
            optimizer.zero_grad()
            loss = ((model(inputs) - inputs) ** 2).mean()
            loss.backward()
            optimizer.step()
    return model




### 5.3 Ventanas e inferencia

`window_data` arma tensores, la clase de la última fila y la máscara totalmente
normal. `reconstruction_errors` calcula el MSE por ventana y `lstm_probabilities`
devuelve las probabilidades multiclase. Solo definen.


In [22]:
def window_data(tensor, rows):
    """Tensor de ventanas, clase de la ultima fila y mascara totalmente normal."""
    xs = tensor[rows]
    last = y.to_numpy()[rows[:, -1]]
    fully_normal = (y.to_numpy()[rows] == normal_idx).all(axis=1)
    return xs, last, fully_normal


def reconstruction_errors(model, xs):
    model.eval()
    errors = []
    loader = DataLoader(TensorDataset(torch.from_numpy(xs)),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            errors.append(((model(batch) - batch) ** 2).mean(dim=(1, 2)).cpu().numpy())
    return np.concatenate(errors) if errors else np.array([])


def lstm_probabilities(model, xs):
    model.eval()
    outputs = []
    loader = DataLoader(TensorDataset(torch.from_numpy(xs)),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    with torch.no_grad():
        for (batch,) in loader:
            logits = model(batch.to(DEVICE))
            outputs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(outputs) if outputs else np.empty((0, len(CLASS_NAMES)))


<a id="sec-metricas"></a>

## 6. Métricas multiclase y cobertura (5.3)

Se separan recall de **detección** (ataque predicho como cualquier ataque) y de
**identificación** (clase de ataque exacta). Las métricas indefinidas quedan en
`None`; la media entre folds usa `ddof=0` e informa folds válidos.


### 6.1 Métricas y utilidades de serialización

`multiclass_metrics` y `attack_metrics` separan recall de detección y de
identificación; `binary_metrics` evalúa el autoencoder; `mean_std` agrega folds;
`json_safe` convierte valores a tipos serializables. Solo definen.


In [23]:
def multiclass_metrics(actual, predicted, probabilities, names):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    labels = np.arange(len(names))
    result = {
        'samples': int(len(actual)),
        'accuracy': float(accuracy_score(actual, predicted)),
        'balanced_accuracy': float(balanced_accuracy_score(actual, predicted)),
        'macro_f1': float(f1_score(actual, predicted, labels=labels,
                                   average='macro', zero_division=0)),
        'classification_report': classification_report(
            actual, predicted, labels=labels, target_names=list(names),
            output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(actual, predicted, labels=labels).tolist(),
        'class_names': list(names),
    }
    if len(np.unique(actual)) == len(names):
        result['roc_auc'] = float(roc_auc_score(
            actual, probabilities, labels=labels, multi_class='ovr', average='macro'))
    else:
        result['roc_auc'] = None
    result.update(attack_metrics(actual, predicted))
    return result


def attack_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    attacks = [index for index in range(len(CLASS_NAMES)) if index != normal_idx]
    detection = {}
    identification = {}
    for index in attacks:
        mask = actual == index
        total = int(mask.sum())
        detection[CLASS_NAMES[index]] = (
            float((predicted[mask] != normal_idx).sum()) / total if total else None)
        identification[CLASS_NAMES[index]] = (
            float((predicted[mask] == index).sum()) / total if total else None)
    normal_mask = actual == normal_idx
    normal_total = int(normal_mask.sum())
    return {
        'detection_recall': detection,
        'identification_recall': identification,
        'normal_fp': (float((predicted[normal_mask] != normal_idx).sum()) / normal_total
                      if normal_total else None),
        'normal_support': normal_total,
    }


def binary_metrics(actual, scores, threshold):
    actual = np.asarray(actual)
    predicted = (np.asarray(scores) > threshold).astype(int)
    result = {'samples': int(len(actual)), 'threshold': float(threshold)}
    if len(np.unique(actual)) == 2:
        result['balanced_accuracy'] = float(balanced_accuracy_score(actual, predicted))
        result['roc_auc'] = float(roc_auc_score(actual, np.asarray(scores)))
        result['recall_attack'] = (
            float((predicted[actual == 1] == 1).mean()) if (actual == 1).any() else None)
        result['fp_normal'] = (
            float((predicted[actual == 0] == 1).mean()) if (actual == 0).any() else None)
    else:
        result['balanced_accuracy'] = None
        result['roc_auc'] = None
        result['recall_attack'] = None
        result['fp_normal'] = None
    return result


def mean_std(values):
    array = np.asarray([value for value in values if value is not None], dtype=float)
    if not len(array):
        return {'mean': None, 'std': None, 'valid_folds': 0}
    return {'mean': float(array.mean()), 'std': float(array.std(ddof=0)),
            'valid_folds': int(len(array))}


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        missing = False
    if isinstance(missing, bool) and missing:
        return None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value


<a id="sec-evaluacion"></a>

## 7. Evaluación de cinco folds con MSE out-of-fold (5.1/5.2)

Para cada fold externo, el híbrido se entrena con MSE generado out-of-fold dentro
del train externo (3 folds internos agrupados). Validation y test externos usan
el componente base ajustado con todo el train externo.


### 7.1 Folds internos (`inner_folds_for`)

Genera 3 folds agrupados dentro del train externo con semilla 42. Se usan para
obtener el MSE out-of-fold del híbrido.


In [24]:
def inner_folds_for(external_train, n_splits=3):
    """Folds internos agrupados dentro del train externo (semilla 42)."""
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    placeholder = np.zeros((len(external_train), 1))
    local = [test for _, test in splitter.split(
        placeholder, y.iloc[external_train], groups=connection_ids[external_train])]
    return [external_train[part] for part in local]




### 7.2 MSE out-of-fold (`oof_hybrid_training_rows`)

Para cada fold interno ajusta el estado, entrena el autoencoder y calcula el MSE
de las filas retenidas; devuelve las features tabulares con `fe_lstm_mse` y sus
etiquetas. Solo define.


In [25]:
def oof_hybrid_training_rows(frame, external_train, include_origin=True):
    """Filas finales del train externo con MSE out-of-fold y su etiqueta (5.2)."""
    parts = []
    for heldout in inner_folds_for(external_train):
        inner_train = np.setdiff1d(external_train, heldout)
        state = fit_feature_state(frame, {'train': inner_train, 'val': heldout,
                                          'test': heldout}, inner_train,
                                  include_origin=include_origin)
        heldout_windows = sequence_indices(heldout)
        train_windows = sequence_indices(inner_train)
        train_xs, _, train_normal = window_data(state['X_tensor'], train_windows)
        model = train_autoencoder(train_xs[train_normal], projection=True)
        mse = reconstruction_errors(model, state['X_tensor'][heldout_windows])
        ends = heldout_windows[:, -1]
        features = state['X'].iloc[ends].copy().reset_index(drop=True)
        features['fe_lstm_mse'] = mse
        features['_row'] = ends
        parts.append(features)
    combined = pd.concat(parts, ignore_index=True).sort_values('_row', kind='stable')
    rows = combined['_row'].to_numpy()
    labels = y.to_numpy()[rows]
    return combined.drop(columns='_row').reset_index(drop=True), labels




### 7.3 Evaluación de una partición (`evaluate_partition`)

Entrena y evalúa XGBoost, LSTM, autoencoder auxiliar e híbrido. Cada modo fallido
se registra como `error` sin inventar métricas. Solo define; la ejecución masiva
ocurre en 7.5.


In [26]:
def evaluate_partition(frame, partitions, name='fold', include_origin=True):
    """Entrena y evalúa los modos supervisados y el auxiliar.

    Un modelo no entrenable o no evaluable se registra como `error`; no se busca
    otro split ni se inventan métricas.
    """
    train_idx, val_idx, test_idx = (partitions[key] for key in ('train', 'val', 'test'))
    result = {'name': name, 'partitions': partitions,
              'coverage': {key: class_counts(rows) for key, rows in partitions.items()},
              'models': {}, 'metrics': {}, 'mse': {}, 'window_rows': {},
              'sequences': {}, 'state': None}
    try:
        state = fit_feature_state(frame, partitions, train_idx,
                                  include_origin=include_origin)
    except Exception as error:
        result['error'] = 'fit_feature_state: ' + str(error)
        print('[' + name + '] no evaluable:', result['error'])
        return result
    state.pop('work', None)
    state.pop('X_stage1', None)
    result['state'] = state
    X, X_tensor = state['X'], state['X_tensor']
    window_rows = {key: sequence_indices(rows) for key, rows in partitions.items()}
    sequences = {key: window_data(X_tensor, rows) for key, rows in window_rows.items()}
    result['window_rows'] = window_rows
    result['sequences'] = sequences

    present_classes = set(y.iloc[train_idx].tolist())
    missing_classes = sorted(set(range(len(CLASS_NAMES))) - present_classes)

    try:
        if missing_classes:
            raise ValueError('train sin clases: ' + str(missing_classes))
        model_xgb = fit_classifier(X.iloc[train_idx], y.iloc[train_idx])
        result['models']['xgb'] = model_xgb
        result['metrics']['xgboost'] = {
            'val': multiclass_metrics(y.iloc[val_idx], model_xgb.predict(X.iloc[val_idx]),
                                      model_xgb.predict_proba(X.iloc[val_idx]), CLASS_NAMES),
            'test': multiclass_metrics(y.iloc[test_idx], model_xgb.predict(X.iloc[test_idx]),
                                       model_xgb.predict_proba(X.iloc[test_idx]), CLASS_NAMES)}
    except Exception as error:
        result['metrics']['xgboost'] = {'error': 'xgboost: ' + str(error)}

    try:
        if missing_classes:
            raise ValueError('train sin clases: ' + str(missing_classes))
        if not len(sequences['train'][0]):
            raise ValueError('sin ventanas de train')
        model_lstm = train_supervised_lstm(sequences, X_tensor.shape[1])
        result['models']['lstm'] = model_lstm
        lstm_val = lstm_probabilities(model_lstm, sequences['val'][0])
        lstm_test = lstm_probabilities(model_lstm, sequences['test'][0])
        result['metrics']['lstm'] = {
            'val': multiclass_metrics(sequences['val'][1], lstm_val.argmax(axis=1),
                                      lstm_val, CLASS_NAMES),
            'test': multiclass_metrics(sequences['test'][1], lstm_test.argmax(axis=1),
                                       lstm_test, CLASS_NAMES)}
    except Exception as error:
        result['metrics']['lstm'] = {'error': 'lstm: ' + str(error)}

    model_ae = None
    try:
        model_ae = train_autoencoder(sequences['train'][0][sequences['train'][2]],
                                     projection=True)
        result['models']['ae'] = model_ae
        mse = {key: reconstruction_errors(model_ae, sequences[key][0])
               for key in sequences}
        result['mse'] = mse
        val_normal = mse['val'][sequences['val'][2]]
        if len(val_normal):
            threshold = float(np.percentile(val_normal, 95))
            result['metrics']['autoencoder'] = {
                'threshold': threshold,
                'val': binary_metrics((sequences['val'][1] != normal_idx).astype(int),
                                      mse['val'], threshold),
                'test': binary_metrics((sequences['test'][1] != normal_idx).astype(int),
                                       mse['test'], threshold)}
        else:
            result['metrics']['autoencoder'] = {
                'error': 'sin ventanas normales de validation',
                'threshold': None, 'val': None, 'test': None}
    except Exception as error:
        result['metrics']['autoencoder'] = {'error': 'autoencoder: ' + str(error)}

    try:
        if missing_classes:
            raise ValueError('train sin clases: ' + str(missing_classes))
        if model_ae is None:
            raise ValueError('autoencoder no entrenado')
        oof_features, oof_labels = oof_hybrid_training_rows(
            frame, train_idx, include_origin=include_origin)
        oof_missing = sorted(set(range(len(CLASS_NAMES))) - set(oof_labels.tolist()))
        if oof_missing:
            raise ValueError('MSE OOF sin clases: ' + str(oof_missing))
        model_hybrid = fit_classifier(oof_features, oof_labels)
        result['models']['hybrid'] = model_hybrid
        hybrid_val_x = X.iloc[window_rows['val'][:, -1]].reset_index(drop=True)
        hybrid_val_x['fe_lstm_mse'] = result['mse']['val']
        hybrid_test_x = X.iloc[window_rows['test'][:, -1]].reset_index(drop=True)
        hybrid_test_x['fe_lstm_mse'] = result['mse']['test']
        result['metrics']['hybrid'] = {
            'val': multiclass_metrics(sequences['val'][1],
                                      model_hybrid.predict(hybrid_val_x),
                                      model_hybrid.predict_proba(hybrid_val_x),
                                      CLASS_NAMES),
            'test': multiclass_metrics(sequences['test'][1],
                                       model_hybrid.predict(hybrid_test_x),
                                       model_hybrid.predict_proba(hybrid_test_x),
                                       CLASS_NAMES)}
    except Exception as error:
        result['metrics']['hybrid'] = {'error': 'hybrid: ' + str(error)}

    print('[' + name + '] ventanas train/val/test:',
          {key: int(len(window_rows[key])) for key in ('train', 'val', 'test')})
    for mode in ('xgboost', 'lstm', 'hybrid'):
        entry = result['metrics'].get(mode, {})
        if 'test' in entry:
            print('  ' + mode, 'test bal_acc/macro_f1/auc:',
                  {key: entry['test'].get(key)
                   for key in ('balanced_accuracy', 'macro_f1', 'roc_auc')})
        else:
            print('  ' + mode, entry.get('error', 'no evaluable'))
    return result




### 7.4 Lectores de métricas (`fold_metric`, `nested_metric`)

Extraen una métrica plana o anidada de un resultado de fold y devuelven `None`
si falta. Solo definen.


In [27]:
def fold_metric(result, mode, split, key):
    entry = result.get('metrics', {}).get(mode)
    if not entry or split not in entry:
        return None
    return entry[split].get(key)


def nested_metric(result, mode, split, kind, attack):
    entry = result.get('metrics', {}).get(mode)
    if not entry or split not in entry:
        return None
    return entry[split].get(kind, {}).get(attack)




### 7.5 Rotación de folds

Activa la evaluación temporal y las ablations, fija `FINAL_FOLD = 0` y ejecuta
`evaluate_partition` para cada fold conservando la lista `rotation`.
**Al ejecutar esta celda se entrenan los modelos de los cinco folds.**


In [28]:
EVAL_TEMPORAL = True
EVAL_ABLATIONS = True

FINAL_FOLD = 0
rotation = []
for k in range(N_SPLITS):
    rotation.append(evaluate_partition(df_raw, fold_partitions(FOLDS, k),
                                       name='fold' + str(k)))


  tasa de bytes: longitudes invalidas excluidas=0


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [21:20:48] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
[fold0] ventanas train/val/test: {'train': 127583, 'val': 29887, 'test': 52357}
  xgboost test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.8869018816537153, 'macro_f1': 0.7772476121602604, 'roc_auc': 0.9889264156943598}
  lstm test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.8488544537618405, 'macro_f1': 0.6103510401970699, 'roc_auc': 0.9744890948871776}
  hybrid test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.7358050785351682, 'macro_f1': 0.6616753547736656, 'roc_auc': 0.9600728072735658}
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
[fold1] ventanas train/val/test: {'train': 144713, 'val': 35227, 'test': 29887}
  xgboost test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.9812222

### 7.6 Liberación de estado

Descarta estado, modelos, secuencias, MSE y ventanas de todos los folds salvo el
candidato (`FINAL_FOLD`). **Modifica `rotation`; no entrena.**


In [29]:
for item in rotation:
    if item['name'] != 'fold' + str(FINAL_FOLD):
        item['state'] = None
        item['models'] = {}
        item['sequences'] = {}
        item['mse'] = {}
        item['window_rows'] = {}



### 7.7 Constantes y agregación (`aggregate_rotation`)

Define las métricas resumidas, los ataques y la función que promedia los folds
(`ddof=0`, folds válidos). Solo define.


In [30]:
ROTATION_METRICS = ('balanced_accuracy', 'macro_f1', 'roc_auc', 'normal_fp')
ATTACKS = ('DoS', 'mitm', 'intrusion')


def aggregate_rotation(results):
    summary = {}
    for mode in ('xgboost', 'lstm', 'hybrid'):
        for split in ('val', 'test'):
            for metric in ROTATION_METRICS:
                values = [fold_metric(item, mode, split, metric) for item in results]
                summary[mode + '.' + split + '.' + metric] = mean_std(values)
            for attack in ATTACKS:
                for kind in ('detection_recall', 'identification_recall'):
                    values = [nested_metric(item, mode, split, kind, attack)
                              for item in results]
                    summary[mode + '.' + split + '.' + kind + '.' + attack] = mean_std(values)
    autoencoder = []
    for item in results:
        entry = item.get('metrics', {}).get('autoencoder')
        autoencoder.append(entry['test'].get('roc_auc')
                           if entry and entry.get('test') else None)
    summary['autoencoder.test.roc_auc'] = mean_std(autoencoder)
    return summary




### 7.8 Resumen de rotación

Calcula e imprime `rotation_summary`. **Ejecutar; no escribe artefactos.**


In [31]:
rotation_summary = aggregate_rotation(rotation)
print('Resumen de rotación (media ± desviación, ddof=0):')
for key, value in rotation_summary.items():
    print('  ' + key, value)


Resumen de rotación (media ± desviación, ddof=0):
  xgboost.val.balanced_accuracy {'mean': 0.9516520740694375, 'std': 0.052696843721529885, 'valid_folds': 5}
  xgboost.val.macro_f1 {'mean': 0.8255719759230142, 'std': 0.04017715190901875, 'valid_folds': 5}
  xgboost.val.roc_auc {'mean': 0.9954097813225957, 'std': 0.004102737572112067, 'valid_folds': 5}
  xgboost.val.normal_fp {'mean': 0.021840037765517302, 'std': 0.0054966668052438905, 'valid_folds': 5}
  xgboost.val.detection_recall.DoS {'mean': 0.9730423937875488, 'std': 0.0063433456494092514, 'valid_folds': 5}
  xgboost.val.identification_recall.DoS {'mean': 0.9725255979219156, 'std': 0.0069399800211894964, 'valid_folds': 5}
  xgboost.val.detection_recall.mitm {'mean': 0.8668658138937563, 'std': 0.21832507047201205, 'valid_folds': 5}
  xgboost.val.identification_recall.mitm {'mean': 0.8662705757985181, 'std': 0.2195119349293212, 'valid_folds': 5}
  xgboost.val.detection_recall.intrusion {'mean': None, 'std': None, 'valid_folds': 0}
 

<a id="sec-temporal"></a>

## 8. Evaluación temporal adicional (5.4)

Cortes cronológicos 60/80 por captura con grupos completos; los grupos que
atraviesan un corte se excluyen y se cuentan. Sin metadata de episodios de ataque
no se afirma generalización entre ataques independientes.


### 8.1 Split temporal (60/80)

Si `EVAL_TEMPORAL`, corta por percentiles 60/80 dentro de cada captura con grupos
completos y evalúa la partición temporal. **Ejecutar; puede entrenar modelos.**


In [32]:
temporal_result = None
temporal_summary = {}
if EVAL_TEMPORAL:
    temporal_splits, temporal_excluded = temporal_partitions(df_raw, connection_ids)
    print('Split temporal -> train/val/test:',
          {key: int(len(value)) for key, value in temporal_splits.items()},
          'grupos excluidos:', len(temporal_excluded))
    if all(len(temporal_splits[key]) for key in ('train', 'val', 'test')):
        temporal_result = evaluate_partition(df_raw, temporal_splits, name='temporal')
        temporal_summary = temporal_result.get('metrics', {})
        for key in ('state', 'models', 'sequences', 'mse', 'window_rows'):
            temporal_result[key] = None if key == 'state' else {}
    else:
        print('Split temporal incompleto; se registra como no evaluable.')



Split temporal -> train/val/test: {'train': 135339, 'val': 41988, 'test': 52683} grupos excluidos: 359
  tasa de bytes: longitudes invalidas excluidas=0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
[temporal] ventanas train/val/test: {'train': 89799, 'val': 32402, 'test': 37402}
  xgboost test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.7646975727229901, 'macro_f1': 0.5918916407000618, 'roc_auc': 0.9724814292743784}
  lstm test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.9655722219000588, 'macro_f1': 0.6723292483489841, 'roc_auc': None}
  hybrid test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.9339970784275433, 'macro_f1': 0.6580979651041117, 'roc_auc': None}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


### 8.2 Ablations de origen y del autoencoder A/B/C

La ablación `sin_origen` repite el fold 0 sin features de origen. Sobre ese estado
se entrenan las variantes A (`X_scaled`), B (`X_tensor`) y C (proyección) y se
resume su desempeño binario. **No reasigna el paquete final.**


In [33]:
ablation_origin = None
ablation_autoencoder = {}
if EVAL_ABLATIONS:
    ablation_origin = evaluate_partition(df_raw, fold_partitions(FOLDS, 0),
                                         name='sin_origen', include_origin=False)
    if ablation_origin.get('state') is not None:
        ablation_state = ablation_origin['state']
        for label, key, projection in (('A', 'X_scaled', False),
                                       ('B', 'X_tensor', False),
                                       ('C', 'X_tensor', True)):
            tensor = ablation_state[key]
            data = {name: window_data(tensor, rows)
                    for name, rows in ablation_origin['window_rows'].items()}
            try:
                model = train_autoencoder(data['train'][0][data['train'][2]], projection)
                val_normal = reconstruction_errors(model, data['val'][0][data['val'][2]])
                if not len(val_normal):
                    ablation_autoencoder[label] = {'evaluated': False,
                                                   'cause': 'sin normal de validation'}
                    continue
                threshold = float(np.percentile(val_normal, 95))
                actual = (data['val'][1] != normal_idx).astype(int)
                scores = reconstruction_errors(model, data['val'][0])
                ablation_autoencoder[label] = {'evaluated': True, **binary_metrics(
                    actual, scores, threshold)}
            except Exception as error:
                ablation_autoencoder[label] = {'evaluated': False, 'cause': str(error)}
    else:
        ablation_autoencoder = {'error': 'ablación de origen no entrenable'}
    if ablation_origin is not None:
        for key in ('state', 'models', 'sequences', 'mse', 'window_rows'):
            ablation_origin[key] = None if key == 'state' else {}
    print('Ablations autoencoder A/B/C:', ablation_autoencoder)


  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
  tasa de bytes: longitudes invalidas excluidas=0
[sin_origen] ventanas train/val/test: {'train': 127583, 'val': 29887, 'test': 52357}
  xgboost test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.8869102457949689, 'macro_f1': 0.7774374723287154, 'roc_auc': 0.9890049039788356}
  lstm test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.8538599076897604, 'macro_f1': 0.6334362595959626, 'roc_auc': 0.9699175864621898}
  hybrid test bal_acc/macro_f1/auc: {'balanced_accuracy': 0.7632431664282107, 'macro_f1': 0.6810034326664953, 'roc_auc': 0.9701224984123811}
Ablations autoencoder A/B/C: {'A': {'evaluated': True, 'samples': 29887, 'threshold': 0.07427985966205597, 'balanced_accuracy': 0.9740863084975429, 'roc_auc': 0.9802539990457898, 'recall_attack': 0.9990156096189003, 'fp_normal': 0.05084299262381454}, 'B': {'evaluated': True, 'samples': 29887,

<a id="sec-exportacion"></a>

## 9. Paquete final y exportación (5.7)

El paquete de referencia conserva el fold externo `k=0` (no el mejor test). No se
reentrena con test. La copia conjunta a `modelos_agente/` la realiza el usuario
tras revisar los resultados.


### 9.1 Selección del fold candidato

Fija `FINAL_FOLD = 0` (no el mejor test), falla si el fold no es entrenable y
recupera modelos, estado, umbral y ventanas del paquete candidato. **Ejecutar tras
la rotación.**


In [34]:
FINAL_FOLD = 0
final = rotation[FINAL_FOLD]
if final.get('error') or final.get('state') is None or not final['models']:
    raise ValueError('El fold final no es entrenable; no hay paquete candidato. '
                     'No se sustituye en silencio por otro fold.')
state = final['state']
model_xgb = final['models']['xgb']
model_lstm = final['models']['lstm']
model_ae = final['models']['ae']
model_hybrid = final['models']['hybrid']
X = state['X']
X_tensor = state['X_tensor']
scaler = state['scaler']
train_means = state['train_means']
LSTM_BASE_COLUMNS = state['base_columns']
TENSOR_COLUMNS = state['tensor_columns']
MISSING_MASK_COLUMNS = state['missing_mask_columns']
all_missing_in_train = state['all_missing_in_train']
known_origins = state['known_origins']
known_topics = state['known_topics']
threshold = final['metrics']['autoencoder']['threshold']
window_rows = final['window_rows']


### 9.2 Matrices finales y directorio de salida

Construye las features híbridas con su `fe_lstm_mse` por partición y define
`OUT_DIR`. Solo prepara datos y directorio.


In [35]:
hybrid = {name: X.iloc[rows[:, -1]].copy().reset_index(drop=True)
          for name, rows in window_rows.items()}
for name in hybrid:
    hybrid[name]['fe_lstm_mse'] = final['mse'][name]

OUT_DIR = '/kaggle/working/modelos_agente'
os.makedirs(OUT_DIR, exist_ok=True)




### 9.3 Helpers de paridad (`sequence_tensor`, `build_parity_payload`)

`sequence_tensor` arma ventanas para el autoencoder. `build_parity_payload`
selecciona ventanas TCP y L2 de test y guarda features, tensor, MSE y etiquetas
de los tres modos para comparar offline/online. Solo definen.


In [36]:
def sequence_tensor(tensor, rows):
    return np.stack([tensor[rows[start:start + SEQ_LEN]]
                     for start in range(len(rows) - SEQ_LEN + 1)])


def build_parity_payload():
    """Muestras TCP y L2 con features/tensores/labels esperados (5.6)."""
    test_rows = final['partitions']['test']
    epochs = df_raw['frame.time_epoch'].to_numpy()
    captures = df_raw['_capture'].to_numpy()
    payload = {}
    for label in ('tcp', 'l2'):
        window = None
        for candidate in window_rows['test']:
            if df_raw.iloc[candidate]['_observation'].eq(label).all():
                window = candidate
                break
        if window is None:
            continue
        end = int(window[-1])
        direction = direction_ids[end]
        members = np.sort(test_rows[direction_ids[test_rows] == direction])
        position = int(np.flatnonzero(members == end)[0])
        sample = members[position - SEQ_LEN:position + 1]
        if len(sample) != SEQ_LEN + 1:
            continue
        capture = captures[end]
        replay = np.flatnonzero((captures == capture) & (epochs <= epochs[end]))
        replay = replay[np.isin(replay, test_rows)]
        replay = replay[np.argsort(epochs[replay], kind='stable')]
        sequences = sequence_tensor(X_tensor, sample)
        mse = reconstruction_errors(model_ae, sequences)
        ends_rows = np.array([sample[SEQ_LEN - 1], sample[SEQ_LEN]])
        hybrid_features = X.iloc[ends_rows].copy().reset_index(drop=True)
        hybrid_features['fe_lstm_mse'] = mse
        payload[label] = {
            'sample_rows': [int(row) for row in sample],
            'replay': [{'row': int(row), **json_safe(df_raw.iloc[row].to_dict())}
                       for row in replay],
            'expected_features': json_safe(X.iloc[sample].to_numpy().tolist()),
            'expected_tensor': json_safe(X_tensor[sample].tolist()),
            'expected_mse': json_safe(mse.tolist()),
            'expected_labels_xgboost': [
                str(value) for value in
                le_target.inverse_transform(model_xgb.predict(X.iloc[sample]))],
            'expected_labels_lstm': [
                CLASS_NAMES[int(index)] for index in
                lstm_probabilities(model_lstm, sequences).argmax(axis=1)],
            'expected_labels_hybrid': [
                CLASS_NAMES[int(index)] for index in
                model_hybrid.predict(hybrid_features)],
        }
    return payload




### 9.4 Escritura de muestras de paridad

**Escribe** `parity_samples.json` en `OUT_DIR`; es el contrato que consume
`tests/run_final_validation.py`.


In [37]:
with open(os.path.join(OUT_DIR, 'parity_samples.json'), 'w') as handle:
    json.dump(json_safe(build_parity_payload()), handle, indent=2, allow_nan=False)



### 9.5 Serialización de modelos y preprocesado

**Escribe** `xgb_case1.ubj`, `xgb_hybrid.ubj`, `lstm_ae.pt`,
`lstm_classifier.pt`, `scaler.joblib` y `le_target.joblib`. `model_ae` y
`model_lstm` pasan a CPU y a modo eval antes de `torch.jit.script`.


In [38]:
model_xgb.save_model(os.path.join(OUT_DIR, 'xgb_case1.ubj'))
model_hybrid.save_model(os.path.join(OUT_DIR, 'xgb_hybrid.ubj'))
model_ae.cpu().eval()
torch.jit.script(model_ae).save(os.path.join(OUT_DIR, 'lstm_ae.pt'))
model_lstm.cpu().eval()
torch.jit.script(model_lstm).save(os.path.join(OUT_DIR, 'lstm_classifier.pt'))
joblib.dump(scaler, os.path.join(OUT_DIR, 'scaler.joblib'))
joblib.dump(le_target, os.path.join(OUT_DIR, 'le_target.joblib'))


['/kaggle/working/modelos_agente/le_target.joblib']

### 9.6 Conjuntos conocidos y estadísticas de faltantes

**Escribe** `known_origins.json`, `known_topics.json` y `nan_statistics.json`,
derivados solo del train del fold candidato.


In [39]:
with open(os.path.join(OUT_DIR, 'known_origins.json'), 'w') as handle:
    json.dump({'version': 1, 'derived_from': 'normal frames in train partition only',
               'key_types': ['ip', 'client'], 'keys': sorted(known_origins)},
              handle, indent=2, ensure_ascii=False)
with open(os.path.join(OUT_DIR, 'known_topics.json'), 'w') as handle:
    json.dump({'version': 1, 'derived_from': 'frames in train partition only',
               'keys': sorted(known_topics)}, handle, indent=2, ensure_ascii=False)
nan_statistics_payload = {
    'version': 1, 'derived_from': 'train partition only',
    'policy': 'train_mean_impute_then_scale_then_zero_missing',
    'nan_features': list(NAN_FEATURES), 'base_columns': list(LSTM_BASE_COLUMNS),
    'tensor_columns': list(TENSOR_COLUMNS),
    'means': {column: float(train_means.fillna(0.0)[column])
              for column in LSTM_BASE_COLUMNS},
    'all_missing_in_train': list(all_missing_in_train),
}
with open(os.path.join(OUT_DIR, 'nan_statistics.json'), 'w') as handle:
    json.dump(nan_statistics_payload, handle, indent=2)


### 9.7 Configuración (`config`) y `pipeline_config.json`

El diccionario agrupa contrato, política de NaN, orden de tensores, etapas de
features, estado de streaming, particiones y artefactos; se guarda en
`pipeline_config.json`. Este bloque largo se conserva íntegro por coherencia del
contrato.


In [40]:
config = {
    'contract_version': 4,
    'dataset': 'MQTT_UAD', 'feature_mode': 'mqtt', 'capture_filter': '',
    'observation_mode': 'ethernet_tcp_and_l2',
    'l2_episode_gap_seconds': L2_EPISODE_GAP_SECONDS,
    'seq_len': SEQ_LEN, 'window_size': SEQ_LEN + 1, 'window_per_flow': True,
    'nan_fill': NAN_FILL, 'threshold': threshold, 'normal_idx': 0,
    'nan_policy': {
        'policy': 'minimal', 'version': 1,
        'statistics_artifact': 'nan_statistics.json',
        'non_observable': 'NaN real en la matriz tabular de XGBoost/hibrido',
        'lstm': 'media de train (0 si falta en train), escalar, 0 en posiciones imputadas',
        'missing_masks': 'missing__<feature> sin escalar para las columnas con NaN',
        'indicator': 'fe_has_mqtt, no sustituye las mascaras de identidad/CONNECT/topic/L2',
        'columns': NAN_FEATURES,
    },
    'known_origins_artifact': 'known_origins.json',
    'known_topics_artifact': 'known_topics.json',
    'nan_statistics_artifact': 'nan_statistics.json',
    'tensor_columns': list(TENSOR_COLUMNS),
    'feature_stages': {
        'stage1_before_split': list(BASE_FEATURE_COLUMNS),
        'stage2_after_split': (['fe_origen_conocido', 'fe_origen_observable']
                               + [f for f, _ in ORIGIN_CONTEXT]
                               + [f for f, _ in L2_CONTEXT]
                               + ['fe_topic_novedad']),
    },
    'known_origin_state': {
        'features': ['fe_origen_conocido', 'fe_origen_observable'],
        'lookup_inputs': ['ip.src', 'mqtt.clientid'],
        'match_rule': '1 if any available prefixed key is in known_origins; else 0',
        'observable_rule': '1 if ip.src or mqtt.clientid is present in the frame; else 0',
        'fit_partition': 'normal frames from train only',
    },
    'class_names': list(CLASS_NAMES),
    'class_index': {name: index for index, name in enumerate(CLASS_NAMES)},
    'label_encoder_artifact': 'le_target.joblib',
    'labels': 'supervised window label = class of its last row',
    'split_strategy': 'tcp_connection_or_l2_episode_grouped_train_val_test',
    'evaluation': {
        'rotation': 'test=fold k, validation=fold (k+1)%5, train=rest; seed 42',
        'final_fold': FINAL_FOLD,
        'hybrid_mse': 'out-of-fold within external train (3 grouped inner folds)',
        'temporal': 'per capture quantiles 60/80, complete groups only',
    },
    'stream_state': {
        'tcp_window_key': ['tcp.stream', 'ip.src', 'tcp.srcport', 'ip.dst', 'tcp.dstport'],
        'l2_episode_pair': ['eth.src', 'eth.dst'],
        'l2_window_direction': ['eth.src', 'eth.dst'],
        'l2_episode_gap_seconds': L2_EPISODE_GAP_SECONDS,
        'l2_state_reset': 'gap strictly greater than threshold or clock rollback',
        'connect_context': {
            'features': ['fe_mqtt_ver', 'fe_mqtt_kalive', 'fe_mqtt_cleansess',
                         'fe_mqtt_clientid_len'],
            'sources': ['mqtt.ver', 'mqtt.kalive', 'mqtt.conflag.cleansess',
                        'mqtt.clientid_len'],
            'method': 'causal forward-fill within the same connection',
            'reset': ('new connection (offline: CONNECT session counter; '
                      'online: new tcp.stream)'),
        },
        'origin_context': {
            'window_seconds': ORIGIN_CONTEXT_WINDOW_SECONDS, 'key': 'ip.src',
            'features': [f for f, _ in ORIGIN_CONTEXT],
            'method': 'causal sliding window over earlier frames in stream order',
            'reset': ('partition and capture/segment start (offline), clock rollback; '
                      'online continuous with per-key rollback'),
        },
        'l2_context': {
            'window_seconds': L2_CONTEXT_WINDOW_SECONDS, 'key': 'eth.src',
            'features': [f for f, _ in L2_CONTEXT],
            'method': 'causal sliding window over earlier frames in stream order',
            'reset': ('partition and capture/segment start (offline), clock rollback; '
                      'online continuous with per-key rollback'),
        },
        'direction_rate': {
            'feature': 'fe_bytes_per_sec', 'transport': '_ctx_bytes_per_s',
            'window_seconds': DIRECTION_RATE_WINDOW_SECONDS,
            'key': 'connection/episode and direction',
            'method': 'sum frame.len of earlier frames in [t-5, t], exclude current, /5, log1p',
            'no_history': 0,
            'reset': 'connection/episode, capture/segment start (partition) or clock rollback',
        },
        'state_retention': {'max_tracked_flows': 4096},
        'topic_novelty': {
            'artifact': 'known_topics.json', 'fit_partition': 'train frames only',
            'rule': '1 if topic observed and not in known_topics; NaN if no topic',
        },
    },
    'feature_columns_case1': list(X.columns),
    'feature_columns_lstm_raw': list(LSTM_BASE_COLUMNS),
    'feature_columns_hybrid': list(hybrid['train'].columns),
}
with open(os.path.join(OUT_DIR, 'pipeline_config.json'), 'w') as handle:
    json.dump(config, handle, indent=2)


### 9.8 Manifiesto de particiones

**Escribe** `evaluation_split.json` con semilla, fold final, particiones,
cobertura y filas de origen.


In [41]:
final_manifest = {
    'seed': SEED, 'final_fold': FINAL_FOLD,
    'partitions': {name: rows.tolist() for name, rows in final['partitions'].items()},
    'coverage': final['coverage'],
    'source_rows': [
        {'capture': capture, 'original_row': int(row)}
        for capture, row in df_raw[['_capture', '_original_row']].itertuples(
            index=False, name=None)
    ],
}
with open(os.path.join(OUT_DIR, 'evaluation_split.json'), 'w') as handle:
    json.dump(final_manifest, handle, indent=2)


### 9.9 Métricas y limitaciones

**Escribe** `metrics.json` con el resumen de rotación, temporal, ablations y
limitaciones declaradas.


In [42]:
metrics_payload = {
    'rotation_summary': rotation_summary,
    'rotation_folds': {item['name']: {'metrics': item['metrics'],
                                      'coverage': item['coverage']}
                       for item in rotation},
    'temporal': temporal_summary,
    'ablation_origin_validation': (ablation_origin['metrics']
                                   if ablation_origin is not None else None),
    'ablation_autoencoder_A_B_C': ablation_autoencoder,
    'limitations': {
        'autoencoder': 'auxiliar binario; su MSE alimenta al hibrido',
        'temporal': 'sin metadata de episodios de ataque no se afirma generalizacion',
        'e2e': 'el denominador e2e es la poblacion seleccionada, no todo el dataset',
    },
}
with open(os.path.join(OUT_DIR, 'metrics.json'), 'w') as handle:
    json.dump(json_safe(metrics_payload), handle, indent=2, allow_nan=False)


### 9.10 Empaquetado ZIP

Reúne los artefactos exportados en `modelos_agente.zip` e imprime la ruta final.
**No copia nada a `modelos_agente/` del repositorio.**


In [43]:
zip_path = '/kaggle/working/modelos_agente.zip'
exported = ('xgb_case1.ubj', 'xgb_hybrid.ubj', 'lstm_ae.pt', 'lstm_classifier.pt',
            'scaler.joblib', 'known_origins.json', 'known_topics.json',
            'nan_statistics.json', 'le_target.joblib', 'pipeline_config.json',
            'metrics.json', 'evaluation_split.json', 'parity_samples.json')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in exported:
        archive.write(os.path.join(OUT_DIR, name), name)
print('Exportación completa:', zip_path)


Exportación completa: /kaggle/working/modelos_agente.zip
